# ⌛️ Chronos-X: Backend Intelligence Visualizer

Welcome to the **Chronos-X Backend Demo**. This notebook bypasses the frontend and interacts directly with the AI forecasting engine.

We will visualize:
1. **The Pulse Vector**: The 3D socio-economic heartbeat of a location.
2. **Spatial Parallel Search**: Finding "Parallel Universes" across India.
3. **AI Forecasting**: Looking into the future with Amazon Chronos.

---

## 1. Setup & Installation
Run this cell to install necessary dependencies if you haven't already.

In [ ]:
%pip install polars matplotlib seaborn torch transformers accelerate

## 2. Initialize the Engine
We load the `HybridPredictor` and inspect the available data range.

In [ ]:
import sys
import os
from pathlib import Path

# Add project root to path so we can import src
project_root = os.getcwd()
if project_root not in sys.path:
    sys.path.append(project_root)

import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from src.ai_forecaster import HybridPredictor

# Initialize Engine
print("⏳ Loading Pulse Vector Database...")
pulse_path = Path("data/output/pulse_vectors.parquet")

if not pulse_path.exists():
    print("⚠️ WARNING: 'data/output/pulse_vectors.parquet' not found!")
    print("   Please run 'python src/data_pipeline.py' first.")
    # Fallback for demo purposes if file missing
    REF_MONTH = "2025-11-01"
    TARGET_MONTH = "2025-12-01"
else:
    predictor = HybridPredictor(model_name="amazon/chronos-t5-tiny") # Use tiny for speed in demo
    predictor.load_pulse_table(str(pulse_path))
    print(f"✅ Engine Ready! Loaded {len(predictor.pulse_table)} data points.")
    
    # DYNAMICALLY DETECT DATES
    # Get sorted list of unique months
    available_months = predictor.pulse_table["month"] .unique().sort().to_list()
    print(f"\n📅 Data range: {[str(m) for m in available_months]}")
    
    if len(available_months) >= 2:
        # Use second to last as reference, last as target
        REF_MONTH = str(available_months[-2])
        TARGET_MONTH = str(available_months[-1])
    elif len(available_months) == 1:
        # If only one month, we can only analyze current state, prediction might fail
        REF_MONTH = str(available_months[0])
        # Fake target for code compatibility
        TARGET_MONTH = str(available_months[0])
        print("⚠️ WARNING: Only 1 month of data found. Predictions may be limited.")
    else:
        print("❌ ERROR: No date columns found.")
    
    print(f"   Selected Reference Month: {REF_MONTH}")
    print(f"   Selected Target Month:    {TARGET_MONTH}")

## 3. The "Parallel Universe" Visualizer
This function visualizes the core hypothesis of Chronos-X: **"The future of Place A is hidden in the history of Place B."**

In [ ]:
def visualize_pincode_intelligence(pincode: str):
    print(f"\n🔍 Analyzing PIN: {pincode}...")
    
    # 1. Run Prediction with DYNAMIC DATES
    try:
        result = predictor.predict(
            pincode, 
            num_parallels=3,
            reference_month=REF_MONTH,
            target_month=TARGET_MONTH
        )
    except Exception as e:
        print(f"❌ Error in prediction: {e}")
        import traceback
        traceback.print_exc()
        return

    if "error" in result:
        print(f"❌ Prediction Result Error: {result['error']}")
        return
        
    # Setup Plot Grid
    fig = plt.figure(figsize=(18, 10), constrained_layout=True)
    gs = fig.add_gridspec(2, 2)
    
    # --- PLOT 1: The Pulse Vector (Radar Chart or Bar) ---
    ax1 = fig.add_subplot(gs[0, 0])
    
    metrics = ['Growth (0-5yr)', 'Flow (Updates)', 'Labor (Bio)']
    
    # Handle potential None values safely
    current_vals = [
        result.get('current', {}).get('growth', 0) or 0,
        result.get('current', {}).get('flow', 0) or 0,
        result.get('current', {}).get('labor', 0) or 0
    ]
    
    # Forecast might be None if using spatial_only or failed
    ens = result.get('ensemble') or {}
    forecast_vals = [
        ens.get('growth', 0) or 0,
        ens.get('flow', 0) or 0,
        ens.get('labor', 0) or 0
    ]
    
    x = np.arange(len(metrics))
    width = 0.35
    
    ax1.bar(x - width/2, current_vals, width, label=f'Current ({REF_MONTH})', color='#3b82f6', alpha=0.8)
    ax1.bar(x + width/2, forecast_vals, width, label=f'Forecast ({TARGET_MONTH})', color='#8b5cf6', alpha=0.8)
    
    ax1.set_ylabel('Normalized Intensity (Z-Score)')
    ax1.set_title(f'Pulse Vector Shift: {pincode} ({result.get("state", "Unknown")})', fontsize=14, pad=10)
    ax1.set_xticks(x)
    ax1.set_xticklabels(metrics)
    ax1.legend()
    ax1.grid(axis='y', linestyle='--', alpha=0.3)

    # --- PLOT 2: Parallel Universes --- 
    ax2 = fig.add_subplot(gs[0, 1])
    
    parallels = result.get("spatial_parallels", {}).get("parallels", [])
    
    # Prepare data for heatmap/comparison
    data = []
    labels = []
    
    # Add Target
    data.append(current_vals)
    labels.append(f"TARGET: {pincode}\n({result.get('state', '')})")
    
    # Add Parallels
    for p in parallels:
        # Fetch parallel's vector to visualize comparison
        # (We need to fetch it manually as it's not in the light result)
        try:
            # Filter needs to match the reference month type (Date or Str)
            # Predictor stores parsed dates, so we might need to convert REF_MONTH back if it's a string
            # But polars filter by string usually works if col is Date
            p_vec = predictor.pulse_table.filter(
                (pl.col("pincode") == p['pincode']) & 
                (pl.col("month") == pl.lit(REF_MONTH).str.strptime(pl.Date, "%Y-%m-%d"))
            )
            
            if len(p_vec) > 0:
                vec = [p_vec['growth_norm'][0], p_vec['flow_norm'][0], p_vec['labor_norm'][0]]
                data.append(vec)
                labels.append(f"MATCH: {p['pincode']}\n({p['state']})\nSim: {p['similarity']:.2f}")
        except Exception as e:
            print(f"Note: Could not fetch parallel data for {p['pincode']}: {e}")
    
    if len(data) > 0:
        data = np.array(data)
        sns.heatmap(data, annot=True, cmap="viridis", xticklabels=metrics, yticklabels=labels, ax=ax2, fmt=".2f")
        ax2.set_title('Spatial Parallel Search Results', fontsize=14, pad=10)
    else:
        ax2.text(0.5, 0.5, "No Parallels Found", ha='center', va='center')
    
    # --- PLOT 3: Forecast Confidence ---
    ax3 = fig.add_subplot(gs[1, :])
    
    # Mock historical trace for demo
    history_x = ['Past', 'Current']
    history_y = [current_vals[0] * 0.9, current_vals[0]] # Mock previous
    
    future_x = ['Future']
    future_y = [forecast_vals[0]]
    
    # Confidence band
    ai_data = result.get("ai_forecast", {})
    if ai_data and isinstance(ai_data, dict) and ai_data.get("uncertainty") and ai_data["uncertainty"] .get("growth_range"):
        lower = ai_data["uncertainty"]["growth_range"][0]
        upper = ai_data["uncertainty"]["growth_range"][1]
    else:
        # Fallback if no AI uncertainty
        lower = forecast_vals[0] - 0.2
        upper = forecast_vals[0] + 0.2

    ax3.plot(history_x + future_x, history_y + future_y, marker='o', linestyle='-', linewidth=2, label='Growth Trend')
    ax3.fill_between(future_x, [lower], [upper], color='gray', alpha=0.2, label='AI Confidence Interval')
    
    ax3.set_title('AI Forecast Trajectory (Growth Signal)', fontsize=14)
    ax3.legend()
    ax3.grid(True, alpha=0.3)

    plt.show()
    
    # Print textual summary
    print("\n✨ AI Reasoning:")
    if parallels:
        print(f"   Based on {len(parallels)} similar regions (like {parallels[0]['state']}),")
    else:
        print("   (No spatial parallels found, relying on pure AI forecast)")
    print(f"   we predict a shift in Growth from {current_vals[0]:.2f} → {forecast_vals[0]:.2f}.")

## 4. Run the Demo
Select a PIN code from the database to analyze.

In [ ]:
# Get a random valid PIN that actually exists in our reference month
try:
    valid_pins_df = predictor.pulse_table.filter(
        pl.col("month") == pl.lit(REF_MONTH).str.strptime(pl.Date, "%Y-%m-%d")
    )
    sample_pins = valid_pins_df["pincode"] .unique().head(5).to_list()
    print(f"Available Sample PINs in {REF_MONTH}: {sample_pins}")

    if sample_pins:
        TARGET_PIN = sample_pins[0] 
        visualize_pincode_intelligence(TARGET_PIN)
    else:
        print(f"❌ No PINs found for month {REF_MONTH}")
except Exception as e:
    print(f"❌ Error selecting PINs: {e}")